# Brain Tumor Detection — Step 1: Baseline CNN

**Goal of this notebook:** get a working end-to-end pipeline — data in, model trained, results evaluated — using a simple CNN built from scratch. This is the *baseline*. Step 2 will swap in transfer learning (EfficientNet/ResNet) to push accuracy up.

**Dataset assumption:** this notebook uses the Kaggle *Brain Tumor MRI Dataset* (masoudnickparvar), which has 4 classes — `glioma`, `meningioma`, `pituitary`, `notumor` — split into `Training/` and `Testing/` folders, each with one subfolder per class.

If you already have a different dataset (e.g. your own Drive folder), just change `train_dir` / `test_dir` in the "Load the data" cell — everything downstream works the same as long as the folder structure is `class_name/image.jpg`.


## 1. Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Get the dataset

Downloads the dataset straight into the Colab runtime via `kagglehub` (no manual upload needed). You'll need a free Kaggle account — the first run will prompt for an API token.


In [ ]:
!pip install -q kagglehub

import kagglehub

dataset_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset downloaded to:", dataset_path)

train_dir = f"{dataset_path}/Training"
test_dir = f"{dataset_path}/Testing"


## 3. Load the data

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Classes:", class_names)


### Sanity check: look at a few samples

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds_raw.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 4. Preprocess

Rescale pixel values to `[0, 1]` and set up caching/prefetching so the GPU isn't waiting on I/O.


In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)
AUTOTUNE = tf.data.AUTOTUNE

def prep(ds, training=False):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prep(train_ds_raw, training=True)
val_ds = prep(val_ds_raw)
test_ds = prep(test_ds_raw)


## 5. Baseline CNN

A small, plain CNN — three conv blocks, a dense head, dropout for regularization. Nothing fancy on purpose: this is the number to beat in Step 2.


In [ ]:
model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
]

EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


### Training curves

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train")
plt.plot(epochs_range, val_acc, label="Val")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train")
plt.plot(epochs_range, val_loss, label="Val")
plt.legend()
plt.title("Loss")

plt.show()


## 7. Evaluate on the held-out test set

This is the number that matters — the test set was never seen during training or validation.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = np.concatenate([y for _, y in test_ds], axis=0)
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))


### Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Baseline CNN")
plt.show()


## 8. Save the model

Saved into the Colab runtime by default. Uncomment the Drive lines if you want it to persist past the session (recommended — Colab runtimes get wiped).


In [ ]:
model.save("brain_tumor_baseline.keras")
print("Saved brain_tumor_baseline.keras")

# Persist to Drive so it survives a runtime reset:
# from google.colab import drive
# drive.mount('/content/drive')
# model.save('/content/drive/MyDrive/brain_tumor_baseline.keras')


## Next: Step 2

This baseline gives you a real accuracy number and a saved model — enough to move on. Step 2 swaps the CNN for a transfer-learning backbone (EfficientNetB0 or ResNet50) plus data augmentation, which should meaningfully beat this baseline. Bring the test accuracy from this run when you're ready for that step, so we can compare.
